# SatQuery AI — Minimal Working Pipeline

This is the real retrieval backbone: **text query → embed → cosine search over image tiles → top match + confidence**.

**Run this in Google Colab** (Runtime → Change runtime type → GPU). It needs real internet access to download model weights, which is why it can't run inside a sandboxed assistant environment — this notebook is written to just work once you paste it into Colab.

Scope on purpose: this proves ONE capability end-to-end (semantic retrieval + a templated grounded answer) on a small image set you provide. That matches what the Feasibility slide promises — don't expand this until this path works cleanly.

## 1. Install dependencies

In [ ]:
!pip install -q open_clip_torch gradio pillow


## 2. Load the encoder

Starts with a general pretrained CLIP (guaranteed to load, works today). If your team has the **RemoteCLIP** checkpoint (fine-tuned for remote sensing — noticeably better on satellite imagery than general CLIP), drop the `.pt` file path into `REMOTECLIP_PATH` below and it will load that instead. Don't block the prototype on getting RemoteCLIP working first — general CLIP is enough to prove the pipeline; swap it in once you have the checkpoint.

In [ ]:
import torch, open_clip

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
REMOTECLIP_PATH = None  # e.g. "/content/RemoteCLIP-ViT-B-32.pt" once you have it

model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
tokenizer = open_clip.get_tokenizer("ViT-B-32")

if REMOTECLIP_PATH:
    state_dict = torch.load(REMOTECLIP_PATH, map_location="cpu")
    model.load_state_dict(state_dict)
    print("Loaded RemoteCLIP checkpoint.")
else:
    print("Using general pretrained CLIP (ViT-B-32, OpenAI weights). Swap in RemoteCLIP later for real accuracy gains on satellite imagery.")

model = model.to(DEVICE).eval()


## 3. Build the image index

Upload a small folder of demo tiles (20-40 images is plenty for a prototype — doesn't need to be the full BigEarthNet-MM archive). Each gets embedded once and cached; queries are then just a cosine similarity lookup, so retrieval itself is near-instant.

In [ ]:
from google.colab import files
import os

os.makedirs("tiles", exist_ok=True)
uploaded = files.upload()  # select your demo image files
for fname in uploaded:
    os.rename(fname, f"tiles/{fname}")
print("Tiles ready:", os.listdir("tiles"))


In [ ]:
from PIL import Image
import torch.nn.functional as F

tile_paths = [f"tiles/{f}" for f in os.listdir("tiles") if f.lower().endswith((".jpg", ".jpeg", ".png"))]

@torch.no_grad()
def embed_images(paths):
    embs = []
    for p in paths:
        img = preprocess(Image.open(p).convert("RGB")).unsqueeze(0).to(DEVICE)
        e = model.encode_image(img)
        embs.append(F.normalize(e, dim=-1))
    return torch.cat(embs, dim=0)

tile_embeddings = embed_images(tile_paths)
print(f"Indexed {len(tile_paths)} tiles.")


## 4. Query function

Embeds the text query, ranks tiles by cosine similarity, and returns the top match with a confidence score. The "grounded answer" here is templated from the query + match score — enough to demo the interaction honestly. Swapping the template for an LLM call (pass the retrieved tile + query as context) is the natural next step once this path is solid, not before.

In [ ]:
@torch.no_grad()
def query(text, top_k=1):
    tokens = tokenizer([text]).to(DEVICE)
    text_emb = F.normalize(model.encode_text(tokens), dim=-1)
    sims = (text_emb @ tile_embeddings.T).squeeze(0)
    scores, idxs = sims.topk(min(top_k, len(tile_paths)))
    results = [(tile_paths[i], float(s)) for s, i in zip(scores, idxs)]
    return results

# quick manual test
for path, score in query("Does this field show crop damage?"):
    print(f"{path}  confidence={score:.3f}")


## 5. Live demo UI (Gradio)

Gives you a shareable public link straight from Colab — good enough to demo live in the internal round without deploying anything.

In [ ]:
import gradio as gr

def gradio_query(text):
    results = query(text, top_k=1)
    if not results:
        return None, "No indexed tiles yet — upload some in step 3."
    path, score = results[0]
    answer = f"Best match: {os.path.basename(path)} (confidence {score:.2f}). Replace this line with an LLM call for a fuller natural-language answer."
    return path, answer

demo = gr.Interface(
    fn=gradio_query,
    inputs=gr.Textbox(label="Ask a question about the archive", placeholder="e.g. Does this field show crop damage?"),
    outputs=[gr.Image(label="Retrieved tile"), gr.Textbox(label="Grounded answer")],
    title="SatQuery AI — Live Prototype"
)
demo.launch(share=True)


## Next steps, in order
1. Get this running end-to-end on ~20 of your own sample tiles first — proves the pipeline, costs almost nothing.
2. Swap in the RemoteCLIP checkpoint (`REMOTECLIP_PATH`) once you have it — same code, better domain accuracy.
3. Replace the templated answer in `gradio_query` with a real LLM call that's given the query + retrieved tile as context.
4. Only after that works: expand toward change detection / optical+SAR fusion, per the roadmap line already in the deck — not before.